# Results Plots

This notebook reads `resutls/*/*.csv` and provides quick summaries and plots for model comparison across cutoffs and datasets.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

results_root = Path('resutls')
csv_files = sorted(results_root.glob('*/*.csv'))
print('results_root:', results_root.resolve())
print('csv files:', len(csv_files))
if len(csv_files) == 0:
    raise FileNotFoundError('No result CSV found under resutls/*/*.csv')

df = pd.concat([pd.read_csv(fp) for fp in csv_files], ignore_index=True)
df['cutoff'] = pd.to_numeric(df['cutoff'], errors='coerce')
print('shape:', df.shape)
display(df.head(10))


In [ ]:
metric = 'f1'  # change metric: f1 / mcc / roc_auc / accuracy
summary = (
    df.groupby(['model', 'cutoff'], as_index=False)[metric]
      .mean()
      .sort_values(['model', 'cutoff'])
)
display(summary.head(20))

plt.figure(figsize=(8, 5))
for model_name in sorted(summary['model'].dropna().unique()):
    sub = summary[summary['model'] == model_name]
    plt.plot(sub['cutoff'], sub[metric], marker='o', label=model_name)
plt.xscale('log', base=2)
plt.xlabel('Cutoff')
plt.ylabel(f'Mean {metric}')
plt.title(f'Model comparison by cutoff ({metric})')
plt.grid(alpha=0.3)
plt.legend()
plt.show()


In [ ]:
metric = 'f1'
best = (
    df.sort_values(metric, ascending=False)
      .groupby('data_name', as_index=False)
      .first()[['data_name', 'model', 'cutoff', metric, 'mcc', 'roc_auc', 'accuracy']]
      .sort_values(metric, ascending=False)
)
display(best.head(30))


In [ ]:
metric = 'f1'
heat = (
    df[df['model'] == 'rf']
      .pivot_table(index='data_name', columns='cutoff', values=metric, aggfunc='mean')
      .sort_index()
)
display(heat.head(20))

plt.figure(figsize=(10, max(6, 0.35 * len(heat))))
im = plt.imshow(heat.values, aspect='auto')
plt.colorbar(im, label=metric)
plt.xticks(range(len(heat.columns)), [str(c) for c in heat.columns], rotation=45)
plt.yticks(range(len(heat.index)), heat.index)
plt.title(f'RF {metric} heatmap by dataset and cutoff')
plt.tight_layout()
plt.show()
